In [ ]:
%load_ext watermark


In [ ]:
import os

from IPython.display import display
from teeplot import teeplot as tp

import pylib  # noqa: F401
from pyfonts import load_google_font


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-02-04-complexity-drivers")
teeplot_subdir


## Example Plot


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.path import Path


# Load Merriweather font
font = load_google_font('Merriweather', weight='regular')

def draw_curved_text_with_arc(ax, text, center_angle, radius, font_size, color='black', spacing=1.3, span_reduction=0.0):
    points_per_data_unit = (8 * 72) / 2.0
    char_width_data = (0.5 * font_size) / points_per_data_unit
    effective_char_width = char_width_data * spacing
    theta_step = effective_char_width / radius
    total_angle = theta_step * (len(text) - 1)
    start_angle = center_angle + total_angle / 2

    for i, char in enumerate(text):
        angle = start_angle - i * theta_step
        x = radius * np.cos(angle)
        y = radius * np.sin(angle)
        rotation = np.rad2deg(angle) - 90
        # Use the custom font here
        ax.text(x, y, char, ha='center', va='center', fontsize=font_size,
                font=font, color=color, rotation=rotation, zorder=20)

    underline_r = radius
    padding_angle = theta_step * 0.5
    start_rad = start_angle + padding_angle - span_reduction
    end_rad = start_angle - total_angle - padding_angle + span_reduction
    theta1 = np.rad2deg(end_rad)
    theta2 = np.rad2deg(start_rad)

    line_width = font_size * 1.435

    arc = patches.Arc((0,0), underline_r*2, underline_r*2, angle=0,
                      theta1=theta1, theta2=theta2, color=color,
                      lw=line_width, alpha=0.15, zorder=5, capstyle='round')
    ax.add_patch(arc)

def draw_bezier_connection(ax, start, end, color='black', linewidth=1, linestyle='-', tension=0.4):
    p0 = start
    p3 = end
    p1 = (start[0] * tension, start[1] * tension)
    p2 = (end[0] * tension, end[1] * tension)

    verts = [p0, p1, p2, p3]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]

    path = Path(verts, codes)
    patch = patches.PathPatch(path, facecolor='none', edgecolor=color, lw=linewidth, ls=linestyle, zorder=1)
    ax.add_patch(patch)

def draw_plot():
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect('equal')
    ax.axis('off')

    limit = 1.0
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit - 0.1, limit + 0.25)

    r = 0.6
    label_r = 0.76
    delta = 12

    c_pheno = '#5CAEAE'
    c_geno = '#6A6AD6'
    c_crypt = '#856AD6'

    ang_p_center = np.deg2rad(145)
    ang_g_center = np.deg2rad(90)
    ang_c_center = np.deg2rad(35)

    ang_p_label = np.deg2rad(158)
    ang_g_label = np.deg2rad(90)
    ang_c_label = np.deg2rad(22)

    ang_nc = np.deg2rad(217.5)
    ang_fitness = np.deg2rad(270)
    ang_drift = np.deg2rad(322.5)

    nodes = {
        'p_early': (r * np.cos(ang_p_center + np.deg2rad(delta)), r * np.sin(ang_p_center + np.deg2rad(delta))),
        'p_all': (r * np.cos(ang_p_center - np.deg2rad(delta)), r * np.sin(ang_p_center - np.deg2rad(delta))),
        'g_early': (r * np.cos(ang_g_center + np.deg2rad(delta)), r * np.sin(ang_g_center + np.deg2rad(delta))),
        'g_all': (r * np.cos(ang_g_center - np.deg2rad(delta)), r * np.sin(ang_g_center - np.deg2rad(delta))),
        'c_early': (r * np.cos(ang_c_center + np.deg2rad(delta)), r * np.sin(ang_c_center + np.deg2rad(delta))),
        'c_all': (r * np.cos(ang_c_center - np.deg2rad(delta)), r * np.sin(ang_c_center - np.deg2rad(delta))),
        'nc': (r * np.cos(ang_nc), r * np.sin(ang_nc)),
        'fitness': (r * np.cos(ang_fitness), r * np.sin(ang_fitness)),
        'drift': (r * np.cos(ang_drift), r * np.sin(ang_drift))
    }

    marker_size = 625
    square_size = marker_size * 0.75
    font_size_top = 36
    font_size_bottom = 31

    bottom_items = [
        ('nc', 'niche\nconst'),
        ('fitness', 'fitness'),
        ('drift', 'drift\n(time)')
    ]

    for name, label in bottom_items:
        x, y = nodes[name]
        ax.scatter(x, y, s=square_size, marker='D', color='black', edgecolors='black', zorder=10, linewidth=2)

        offset_y = -0.07
        if '\n' in label:
             offset_y = -0.09

        # Apply font to bottom labels
        ax.text(x, y + offset_y, label, ha='center', va='top', fontsize=font_size_bottom, font=font, color='black')

    pairs = [
        ('p_early', 'p_all', 'Phenotype', ang_p_label, c_pheno, 0.0),
        ('g_early', 'g_all', 'Genotype', ang_g_label, c_geno, 0.04),
        ('c_early', 'c_all', '(+Cryptic)', ang_c_label, c_crypt, 0.04)
    ]

    for early, all_n, label, ang, color, span_red in pairs:
        ax.scatter(*nodes[early], s=marker_size, marker='o', facecolor='white', edgecolor=color, zorder=10, linewidth=2.5)
        ax.scatter(*nodes[all_n], s=marker_size, marker='o', facecolor=color, edgecolor=color, zorder=10, linewidth=2.5)
        draw_curved_text_with_arc(ax, label, ang, label_r, font_size_top, color=color, spacing=1.3, span_reduction=span_red)

    nc_pheno_start = (nodes['nc'][0], nodes['nc'][1] + 0.025)
    nc_geno_start = (nodes['nc'][0], nodes['nc'][1] - 0.025)

    thick_lw = 10.4
    mid_lw = 5.2

    draw_bezier_connection(ax, nc_pheno_start, nodes['p_early'], color=c_pheno, linewidth=1.5, linestyle='--', tension=0.4)
    draw_bezier_connection(ax, nc_pheno_start, nodes['p_all'], color=c_pheno, linewidth=thick_lw, linestyle='-', tension=0.4)

    draw_bezier_connection(ax, nc_geno_start, nodes['g_early'], color=c_geno, linewidth=thick_lw, linestyle='-', tension=0.4)
    draw_bezier_connection(ax, nc_geno_start, nodes['g_all'], color=c_geno, linewidth=1.5, linestyle='--', tension=0.4)

    draw_bezier_connection(ax, nodes['drift'], nodes['c_early'], color=c_crypt, linewidth=1.5, linestyle='--', tension=0.4)
    draw_bezier_connection(ax, nodes['drift'], nodes['c_all'], color=c_crypt, linewidth=thick_lw, linestyle='-', tension=0.4)

    # --- Legend ---
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='w', markeredgecolor='black', markeredgewidth=2.6, markersize=20, label='early'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markeredgecolor='black', markeredgewidth=2.6, markersize=20, label='all'),
        Line2D([0], [0], color='black', lw=2.6, linestyle='--', label='p < 0.05'),
        Line2D([0], [0], color='black', lw=2.6, linestyle='-', label='p < 0.01'),
        Line2D([0], [0], color='black', lw=mid_lw, label='$R^2 = 0.2$'),
        Line2D([0], [0], color='black', lw=thick_lw, label='$R^2 = 0.4$')
    ]

    font2 = font.copy()
    font2.set_size(23)
    ax.legend(handles=legend_elements, loc='upper center',
                    bbox_to_anchor=(0.5, 1.05), # Moved up slightly
                    ncol=3, frameon=False,
                    columnspacing=1.5,   # Space between columns
                    handletextpad=0.5,   # Space between symbol and text
                    labelspacing=1.5,    # THIS controls space between rows
                    prop=font2)           # Font object with size 45 set earlier

    plt.tight_layout()

# draw_plot()


In [ ]:
tp.tee(
    draw_plot,
    teeplot_subdir=teeplot_subdir,
)
